In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

from ingest import load_faq_data
documents = load_faq_data(path="../data/studygrid_faq.json")

doc_idx = {}
for doc in documents:
    doc_idx[doc["id"]] = doc

In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

from retriever import Retriever
from agent import GridMindAgent
from prompts import INSTRUCTIONS, USER_PROMPT_TEMPLATE
from evaluation_utils import GridMindAgentWithUsage

retriever = Retriever(
    documents,
    instructions=INSTRUCTIONS,
    prompt_template=USER_PROMPT_TEMPLATE
)

base_agent = GridMindAgent(
    retriever=retriever,
    openai_client=openai_client
)

assistant = GridMindAgentWithUsage(base_agent)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.chat(question)
answer_llm

iteration #1...
function_call: search {"query":"What does StudyGrid do for students?"}
iteration #2...


'StudyGrid is a mobile app designed to streamline a student’s academic life. It brings together:\n\n- **Group chat** so classmates can communicate easily.  \n- **Class materials** (notes, PDFs, links, etc.) shared in one place.  \n- **Shared and personal to‑do lists** that let students keep track of assignments and tasks.  \n- **Smart notifications** that alert users to new messages, uploaded files, and assigned tasks.\n\nAll of these features are integrated into a single app, helping students stay organized, collaborate, and stay on top of their coursework.'

In [5]:
assistant.total_cost()

0.000171525

In [6]:
doc_id = rec["document"]
answer_orig = doc_idx[doc_id]["answer"]
answer_orig

'StudyGrid is a mobile app that combines group chat, class materials, shared and personal to-do lists, and smart notifications — all in one place for students.'

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]

    answer_llm = assistant.chat(question)
    answer_orig = doc_idx[doc_id]["answer"]

    return {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
        "trajectory": assistant.last_trajectory,   
    }

In [8]:
record = generate_rag_answer(rec)
record

iteration #1...
function_call: search {"query":"What does StudyGrid do for students"}
iteration #2...


{'question': 'Can you tell me what StudyGrid actually does for students?',
 'answer_llm': 'StudyGrid is a mobile app that helps students stay organized and connected by bringing several key study‑support tools into one place:\n\n* **Group chat** – Communicate with classmates and group members in real time.  \n* **Class materials** – Share and access course files and resources within the group.  \n* **Shared & personal to‑do lists** – Create, view, and track tasks that are shared with the group or kept private.  \n* **Smart notifications** – Get alerts for new chat messages, uploaded files, and tasks assigned to you.\n\nAll of these features are designed to streamline collaboration and keep students on top of their coursework.',
 'answer_orig': 'StudyGrid is a mobile app that combines group chat, class materials, shared and personal to-do lists, and smart notifications — all in one place for students.',
 'document': 1,
 'trajectory': [{'tool': 'search',
   'arguments': '{"query":"What d

In [9]:
assistant.reset_usage()
assistant.total_cost()  

0.0

In [12]:
def generate_rag_answer_safe(rec, max_retries=2):
    for attempt in range(max_retries):
        try:
            return generate_rag_answer(rec)
        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    "question": rec["question"],
                    "answer_llm": None,
                    "answer_orig": doc_idx[rec["document"]]["answer"],
                    "document": rec["document"],
                    "trajectory": [],
                    "error": str(e),
                }

In [10]:
from tqdm.auto import tqdm

In [13]:
gt_rag_results = []

for rec in tqdm(ground_truth):
    result = generate_rag_answer_safe(rec)
    gt_rag_results.append(result)

  0%|          | 0/170 [00:00<?, ?it/s]

iteration #1...
function_call: search {"query":"StudyGrid what it does for students"}
iteration #2...
iteration #1...
function_call: search {"query":"StudyGrid just a chat app or does it have other tools?"}
iteration #2...
iteration #1...
function_call: search {"query":"StudyGrid mobile app features"}
iteration #2...
iteration #1...
function_call: search {"query":"StudyGrid class materials to-do lists together"}
iteration #2...
iteration #1...
function_call: search {"query":"StudyGrid keep me notified about school stuff notification system"}
iteration #2...
iteration #1...
function_call: search {"query":"sign up for StudyGrid steps"}
iteration #2...
iteration #1...
function_call: search {"query":"What information do I need to provide when creating a StudyGrid account?"}
iteration #2...
iteration #1...
function_call: search {"query":"Sign Up button app StudyGrid sign up button location"}
iteration #2...
iteration #1...
function_call: search {"query":"Do I need a phone number to register

In [14]:
failed = [r for r in gt_rag_results if r.get("error") is not None]
len(failed)

93

In [ ]:
for r in failed:
    print(r["question"], "-", r["error"])

In [ ]:
succeeded = [r for r in gt_rag_results if r.get("error") is None]
len(succeeded)  